In [ ]:
!pip install -U transformers datasets evaluate accelerate torch torchvision pillow

from transformers import pipeline
from datasets import load_dataset
import evaluate
import time
import numpy as np

In [ ]:
import torch
print(torch.cuda.is_available())  # should be True

# load model
clf = pipeline(
    "text-classification",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    device=0,              # <-- use GPU
    truncation=True
)

# load dataset
dataset = load_dataset("imdb", split="test[:200]")
texts = list(dataset["text"])
labels = list(dataset["label"])

# run prediction
preds = clf(texts, batch_size=8, truncation=True)

pred_labels = [1 if p["label"] == "POSITIVE" else 0 for p in preds]

# evaluate
accuracy = evaluate.load("accuracy")
acc = accuracy.compute(predictions=pred_labels, references=labels)

print("Accuracy:", acc)

# latency
def avg_latency(model, inputs, n=50):
    t0 = time.time()
    for i in range(n):
        model(inputs[i], truncation=True)
    return (time.time() - t0) / n

lat = avg_latency(clf, texts)

print("Avg latency per example:", lat)


True


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

...

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Accuracy: {'accuracy': 0.885}
Avg latency per example: 0.0049938344955444336


In [ ]:
# Task 2: Zero-shot Classification

zsc = pipeline("zero-shot-classification")

result = zsc(
    "This course is about machine learning and AI.",
    candidate_labels=["education", "sports", "politics"]#pick among three candidate labels, no tuning
)#defaults to facebook/bart-large-mnli

print(result)

No model was supplied, defaulted to facebook/bart-large-mnli and revision d7645e1.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

{'sequence': 'This course is about machine learning and AI.', 'labels': ['education', 'sports', 'politics'], 'scores': [0.520648717880249, 0.2945020794868469, 0.18484920263290405]}


In [ ]:
# Model Comparison
clf1 = pipeline(
    "text-classification",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    device=0,              # <-- use GPU
    truncation=True
)

clf2 = pipeline(
    "text-classification",
    model="textattack/bert-base-uncased-imdb",
    device=0,              # <-- use GPU
    truncation=True
)

# evaluate both model
def evaluate_model(model, texts, labels):
    preds = model(texts, batch_size=8, truncation=True)
    pred_labels = [1 if p["label"] == "POSITIVE" else 0 for p in preds]

    acc = accuracy.compute(predictions=pred_labels, references=labels)["accuracy"]
    lat = avg_latency(model, texts)

    return acc, lat

acc1, lat1 = evaluate_model(clf1, texts, labels)
acc2, lat2 = evaluate_model(clf2, texts, labels)

# results table
print("Model Comparison:")
print(f"Model 1 (DistilBERT) -> Accuracy: {acc1:.4f}, Latency: {lat1:.4f}")
print(f"Model 2 (BERT IMDb) -> Accuracy: {acc2:.4f}, Latency: {lat2:.4f}")

# failure case
example = "The movie was not bad at all"

print("Model 1:", clf1(example))
print("Model 2:", clf2(example))

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/511 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Model Comparison:
Model 1 (DistilBERT) -> Accuracy: 0.8850, Latency: 0.0030
Model 2 (BERT IMDb) -> Accuracy: 1.0000, Latency: 0.0052
Model 1: [{'label': 'POSITIVE', 'score': 0.9973582625389099}]
Model 2: [{'label': 'LABEL_1', 'score': 0.7917779684066772}]


Model 2 (BERT IMDb) achieves higher accuracy (1.000) compared to Model 1 (DistilBERT, 0.885), but it has higher latency. This suggests a tradeoff between performance and efficiency. DistilBERT is a compressed model designed for faster inference, while BERT is larger and better tuned for the IMDB dataset, which explains its superior accuracy. However, BERT also produces less interpretable labels (LABEL_0/1), which may require additional processing. A failure case observed is that both models handle ambiguous sentiment differently, indicating sensitivity to phrasing and dataset-specific training.